# Time series
---  
Comparisons of model outputs with SNOTEL site records
  
*J. Michelle Hu  
University of Utah  
Aug 2024  
Updated April 2025*  


In [ ]:
import os
import sys

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import pyproj
import geopandas as gpd
import xarray as xr
import seaborn as sns

from pathlib import PurePath

sys.path.append('/uufs/chpc.utah.edu/common/home/u6058223/git_dirs/env/')
import helpers as h

sys.path.append('/uufs/chpc.utah.edu/common/home/u6058223/git_dirs/ucrb-isnobal/scripts/')
import processing as proc

# Set seaborn palette
sns.set_palette('icefire')

In [ ]:
%reload_ext autoreload
%autoreload 2

### Env setup

In [ ]:
# Locate pyproj_datadir for studio env
# From https://stackoverflow.com/questions/69630630/on-fresh-conda-installation-of-pyproj-pyproj-unable-to-set-database-path-pypr
CONDA_ENV = 'studio'
miniconda_dir = '/uufs/chpc.utah.edu/common/home/u6058223/software/pkg/miniconda3'
proj_version = h.fn_list(miniconda_dir, f'envs/{CONDA_ENV}/conda-meta/proj-[0-9]*.json')[0]

VERSION = PurePath(proj_version).stem
pyprojdatadir = f'{miniconda_dir}/pkgs/{VERSION}/share/proj'
print(pyprojdatadir)
pyproj.datadir.set_data_dir(pyprojdatadir)

### Directories and global variables

In [ ]:
workdir = '/uufs/chpc.utah.edu/common/home/skiles-group3/model_runs/'
script_dir = '/uufs/chpc.utah.edu/common/home/skiles-group3/jmhu/isnobal_scripts'
poly_dir = '/uufs/chpc.utah.edu/common/home/skiles-group3/jmhu/ancillary/polys'
aso_dir = '/uufs/chpc.utah.edu/common/home/skiles-group3/ASO'

# SNOTEL all sites geojson fn - snotel site json
allsites_fn = '/uufs/chpc.utah.edu/common/home/skiles-group3/ancillary_sdswe_products/SNOTEL/snotel_sites_32613.json'

# nwm proj4 file
proj_fn = "/uufs/chpc.utah.edu/common/home/skiles-group3/jmhu/ancillary/NWM_datasets_proj4.txt"

In [ ]:
# Basin-specific variables
# basin = 'yampa'
basin = 'animas'
# basin = 'blue'
# basin = 'erw'

# Span of water years
WYs = [2021, 2022, 2023, 2024]

# Update basindirs
basindirs = h.flatten([h.fn_list(workdir, f'{basin}*/wy{WY}/{basin}*/') for WY in WYs])
basindirs

In [ ]:
# Figure out filenames
poly_fn = h.fn_list(script_dir, f'*{basin}*setup/polys/*shp')[0]
print(poly_fn)

### Specify variables of interest
- calculate density or SWE in separate script and output
- use this space for visualization only

In [ ]:
var = 'density'
compute_density = False
compute_SWE = False
# pull depth only
if var == 'depth':
    snowvar = 'SNOWDEPTH'
    NWM_var = 'SNOWH'
    UA_var = 'DEPTH'
    thisvar = 'thickness'
    aso_var = 'snowdepth'
    band_name = 'snow_depth'
elif var == 'SWE':
    compute_SWE = True
    # pull SWE only
    snowvar = 'SWE'
    NWM_var = 'SNEQV'
    UA_var = 'SWE'
    thisvar = 'SWE'
    aso_var = 'swe'
    band_name = 'swe'
else:
    snowvar = 'both' # change this to both to pull depth and SWE from get_snotel()
    NWM_var = var.upper()
    UA_var = var.upper()
    compute_density = False # I've already done this
    thisvar = 'snow_density'
    aso_var = var
    band_name = var

### SNOTEL extraction and point specification
- identify SNOTEL sites within the specified basin
- extract site metadata (site name, site number, and coordinates)
- extract snow depth values for WY of interest

In [ ]:
# Locate SNOTEL sites within basin
found_sites = proc.locate_snotel_in_poly(poly_fn=poly_fn, site_locs_fn=allsites_fn, buffer=200)

# Get site names and site numbers
sitenames = found_sites['site_name']
sitenums = found_sites['site_num']
print(sitenames)

ST_arr = ['CO'] * len(sitenums)
gdf_metloom, snotel_dfs, snotel_metadf = proc.get_snotel(sitenums, sitenames, ST_arr, WY=WYs, return_meta=True, snowvar=snowvar)
gdf_metloom

### NWM
Processed using extract_nwm_timeseries_point.py blue 2019 script to minimize extraction and transformation

In [ ]:
# NWM SWE is in mm, convert
nwm_dir = '/uufs/chpc.utah.edu/common/home/skiles-group3/ancillary_sdswe_products/NWM/'
nwm_df_list = []
for WY in WYs:
    if WY > 2022:
        continue
    else:
        # Check if the csv already exists
        if len(h.fn_list(nwm_dir, f'{basin}/*{NWM_var}*{WY}.csv')) > 0:
            nwm_csv = h.fn_list(nwm_dir, f'{basin}/*{NWM_var}*{WY}.csv')[0]
            print(nwm_csv)
            nwm_df = pd.read_csv(nwm_csv, index_col=0)
            nwm_df.index.name = 'Date'

            # Set as DatetimeIndex
            nwm_df.index = pd.to_datetime(nwm_df.index)
            nwm_daily_df = nwm_df.resample('1D').mean()

            if NWM_var == 'SNEQV':
                nwm_daily_df = nwm_daily_df * 0.001
            elif NWM_var == 'DENSITY':
                # Modify for the NWM SWE adjustment above
                nwm_daily_df = nwm_daily_df * 1000  # Convert to kg m-3
            nwm_df_list.append(nwm_daily_df)
        else:
            if NWM_var == 'DENSITY':
                # Pull depth and SWE from NWM and calculate density
                nwm_depth_csv = h.fn_list(nwm_dir, f'{basin}/*SNOWH*{WY}.csv')[0]
                nwm_SWE_csv = h.fn_list(nwm_dir, f'{basin}/*SNEQV*{WY}.csv')[0]
                nwm_mini_df_list = []
                # Load the depth and SWE csvs
                for nwm_csv in [nwm_depth_csv, nwm_SWE_csv]:
                    nwm_df = pd.read_csv(nwm_csv, index_col=0)
                    nwm_df.index.name = 'Date'

                    # Set as DatetimeIndex
                    nwm_df.index = pd.to_datetime(nwm_df.index)
                    nwm_daily_df = nwm_df.resample('1D').mean()

                    if NWM_var == 'SNEQV':
                        nwm_daily_df = nwm_daily_df * 0.001
                    nwm_mini_df_list.append(nwm_daily_df)
                    big_list.append(nwm_mini_df_list)
                # Now calculate the density
                # density - put this into kg m-3 units, modifying for the NWM SWE adjustment above
                nwm_daily_df = nwm_mini_df_list[-1] / nwm_mini_df_list[0] * 1000
                nwm_df_list.append(nwm_daily_df)
                # save this to file (because it doesn't otherwise exist)
                # in the same format as the extracted NWM files output from extract_nwm_timeseries_point.py
                nwm_daily_df.to_csv(f'{nwm_dir}/{basin}/{basin}_nwm_snotelmetloom_{NWM_var}_wy{WY}.csv')
                # else:
                #     # Load it
                #     nwm_daily_df = pd.read_csv(nwm_csv[0], index_col=0)
            else:
                print(f'NWM csv not found, please run extract_nwm_timeseries_point.py {basin} {WY} -var {NWM_var}')

    # Concatenate the df_list into single df by index
    nwm_df = pd.concat(nwm_df_list, axis=0)
nwm_df

In [ ]:
# Find how many values are unreasonable
nwm_df.where(nwm_df > 1000).plot(linewidth=0, marker='.')
# And clean it up
nwm_df = nwm_df.where(nwm_df < 1000, other=np.nan)
nwm_df = nwm_df.dropna(how='all')
fig, ax = plt.subplots(figsize=(8,4))
(nwm_df).plot(ax=ax)
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')

### UA

In [ ]:
# UA dir
ua_dir = '/uufs/chpc.utah.edu/common/home/skiles-group3/ancillary_sdswe_products/UA'

if UA_var == 'DEPTH':
    dropvar = 'SWE'
elif UA_var == 'SWE':
    dropvar = 'DEPTH'
else:
    dropvar = False

ua_df_list = []
for WY in WYs:
    # Establish filename of time series based on this WY (netcdf much smaller than csv)
    ua_ts_fn = f'{ua_dir}/{basin}/{basin}_ua_800m_snotelmetloom_{UA_var}_wy{WY}.csv'
    print(ua_ts_fn)

    # If this file does not exist, process
    if not os.path.exists(ua_ts_fn):
        print(f'UA file not found, please run extract_ua_timeseries_point.py {basin} {WY} -var {UA_var}')
    else:
        ua_datadf = pd.read_csv(ua_ts_fn, index_col=0)
        ua_datadf.index.name = 'Date'
        # Set as DatetimeIndex
        ua_datadf.index = pd.to_datetime(ua_datadf.index)
        ua_df_list.append(ua_datadf)

# Concatenate the df_list into single df by index
ua_df = pd.concat(ua_df_list, axis=0)
ua_df

In [ ]:
fig, ax = plt.subplots(figsize=(8,4))
ua_df.plot(ax=ax)
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')

### iSnobal output extraction

- iSnobal time decay
- iSnobal HRRR-MODIS

In [ ]:
figsize = (18, 4)
linestyles = ['-', '--']
linewidth = 1
marker = None
color = 'k'
snotelcolors = ['dodgerblue', 'gray']
isnobalcolors = ['k', 'coral']
# labels = ['iSnobal-HRRR', 'HRRR-MODIS']
# labels = ['HRRR-MODIS'] # This is acutally HRRR-SPIReS, just not updated in the snotel extraction script output yet
labels = ['HRRR-SPIReS'] # extracted at least for snow_density
# labels = ['iSnobal-HRRR']

In [ ]:
basindirs = h.fn_list(workdir, f'{basin}*/wy*/{basin}*solar_albedo/')
basindirs

In [ ]:
%%time
# Takes about 2.5 min
chunks = 'auto'

# currently for one WY (only one per basin as of 20240618), will need to add more for WYs in the future
month = 'run20'
varname = 'snow'

# snow.nc contains these variables as well that we will drop
# depth only
if thisvar == 'thickness':
    drop_var_list = ['snow_density', 'specific_mass', 'liquid_water', 'temp_surf', 'temp_lower', 'temp_snowcover', 'thickness_lower', 'water_saturation', 'projection']
elif thisvar == 'SWE':
    # requires calculation of SWE, keep depth and density
    drop_var_list = ['specific_mass', 'liquid_water', 'temp_surf', 'temp_lower', 'temp_snowcover', 'thickness_lower', 'water_saturation', 'projection']
else:
    # density only
    drop_var_list = ['thickness', 'specific_mass', 'liquid_water', 'temp_surf', 'temp_lower', 'temp_snowcover', 'thickness_lower', 'water_saturation', 'projection']

ds_dict = dict()
df_list = []
for kdx, label in enumerate(labels):
    # specify basindir based on the label

    for WY in WYs:
        if thisvar == 'SWE':
            # account for extraction with specific_mass
            model_ts_fn = f'{PurePath(basindirs[0]).parents[1]}/wy{WY}/{basin}_{label}_specific_mass_snotelmetloom_wy{WY}.csv'
        else:
            model_ts_fn = f'{PurePath(basindirs[0]).parents[1]}/wy{WY}/{basin}_{label}_{thisvar}_snotelmetloom_wy{WY}.csv'
        print(model_ts_fn)

        if not os.path.exists(model_ts_fn):
            print(f'Model file not found, please run extract_isnobal_timeseries_point.py {basin} {WY} -var {var} -label {label}')
            continue
        else:
            df = pd.read_csv(model_ts_fn, index_col=0)
            df.index.name = 'Date'
            # Set as DatetimeIndex
            df.index = pd.to_datetime(df.index)
            df_list.append(df)

    isnobal_df = pd.concat(df_list, axis=0)
    ds_dict[f'{label}_{thisvar}'] = isnobal_df

### Individual iSnobal line plots

In [ ]:
%%time
colors = ['cornflowerblue', 'xkcd:coral']
linestyles = ['-', '-']
linewidths = [1, 1]
alphas = [1, 1]
markersize = 1
snotelcolor = 'dimgrey'
snotel_lw = 0.5
fig, axes = plt.subplots(len(sitenames), 1, figsize=(8, int(2*len(sitenames))), sharex=True)
for kdx, sitename in enumerate(sitenames):
    try:
        ax = axes[kdx]
        # for mdx, label in enumerate(labels):
        for mdx, key in enumerate(ds_dict.keys()):
            label = key.split(f'_{thisvar}')[0]
            snow_var_data = ds_dict[f'{label}_{thisvar}']
            snow_var_data[sitename].plot(ax=ax, color=colors[mdx],
                                            label=label, linewidth=linewidths[mdx],
                                            linestyle=linestyles[mdx], alpha=alphas[mdx]
                                        )

        # Plot WY time series of snow depth
        if thisvar == 'thickness':
            (snotel_dfs[f'{sitename}']['SNOWDEPTH_m']).plot(ax=ax, label=f'{sitename} Snow Depth [m]',
                                                            linestyle='-', linewidth=snotel_lw, color=snotelcolor,
                                                            marker='.', markersize=markersize
                                                        )
            ax.set_ylim(0, 3)
            ax.set_ylabel('Snow Depth [m]')
        else:
            if snowvar == 'SWE':
                (snotel_dfs[f'{sitename}']['SWE_m']).plot(ax=ax, label=f'{sitename} SWE [m]',
                                                                linestyle='-', linewidth=snotel_lw, color=snotelcolor,
                                                                marker='.', markersize=markersize
                                                            )
                ax.set_ylim(0, 1.5)
                ax.set_ylabel('SWE [m]')
            else:
                (snotel_dfs[f'{sitename}']['SNOWDENSITY_kgm3']).plot(ax=ax, label=f'{sitename} density [kgm-3]',
                                                                linestyle='-', linewidth=snotel_lw, color=snotelcolor,
                                                                marker='.', markersize=markersize
                                                            )
                ax.set_ylabel('Snow Density [kgm3]')
        ax.set_xlabel('')
        ax.annotate(sitename, xy=(0.025, 0.85), xycoords='axes fraction', ha='left', fontsize=10, fontstyle='italic')
        if kdx == 0:
            ax.legend(['HRRR-SPIReS', 'SNOTEL'], bbox_to_anchor=(1,1), frameon=False)
        else:
            # turn off legend
            ax.legend().set_visible(False)
    except:
        print(f'{sitename} error')
        pass
# plt.suptitle(PurePath(basindirs[0]).parents[1].name)
plt.tight_layout()


### ASO value extraction

In [ ]:
# TODO add a fix for detecting the correct ASO file given all of the naming systems --> maybe symlink to a standardized name?

In [ ]:
gdf_snotel = gpd.GeoDataFrame(data=np.array([sitenames, sitenums]).T, columns=['sitenames', 'sitenums'], geometry=gdf_metloom.geometry)
gdf_snotel

In [ ]:
%%time
noaso = False
buffer = False
r = np.sqrt(100**2 * 2)
if noaso:
    pass
else:
    aso_df_list = []
    for WY in WYs:
        # Locate ASO snow var files
        state = 'CO'
        # from https://nsidc.org/sites/nsidc.org/files/technical-references/ASO_Basins.pdf
        # only blue river basin has different names
        basin_name = basin.capitalize() # more recent collections
        if compute_density:
            # Read in snow depth and swe files (these should be co-located at output at the same res)
            depth_fns = h.fn_list(aso_dir, f'{state}/*{basin_name}*{WY}*snowdepth*tif')
            swe_fns = h.fn_list(aso_dir, f'{state}/*{basin_name}*{WY}*swe*tif')
            if len(depth_fns) == 0:
                depth_fns = h.fn_list(aso_dir, f'{state}/*/*{basin_name}*{WY}*snowdepth*tif')
                swe_fns = h.fn_list(aso_dir, f'{state}/*/*{basin_name}*{WY}*swe*tif')

            # Load snow var arrays and squeeze out single dimensions
            depth_list = [np.squeeze(xr.open_dataset(fn, chunks=chunks)) for fn in depth_fns]
            swe_list = [np.squeeze(xr.open_dataset(fn, chunks=chunks)) for fn in swe_fns]

            # Rename band_data to be more descriptive
            depth_list = [ds.rename_vars({'band_data': 'snow_depth'}) for ds in depth_list]
            swe_list = [ds.rename_vars({'band_data': 'swe'}) for ds in swe_list]

            # Compute density here, convert to dataset and rename band_data
            snow_var_list = [swe['swe'] / depth['snow_depth'] * 1000 for depth, swe in zip(depth_list, swe_list)]
            snow_var_list = [xr.Dataset({f'{band_name}': snow_var_da}) for snow_var_da in snow_var_list]

            # Deal with adding time input for ASO data by snagging from depth fns
            inputvar = '_snowdepth'
            snow_var_list = [np.squeeze(proc.assign_dt(ds, proc.extract_dt(fn, inputvar=inputvar))) for ds, fn in zip(snow_var_list, depth_fns)]

            # Get dates, could easily just pull from filenames, but this is fine
            date_list = [proc.extract_dt(fn, inputvar=inputvar)[0] for fn in depth_fns]
        else:
            # Water year collections should all be post January so this should work
            snow_var_fns = h.fn_list(aso_dir, f'{state}/*{basin_name}*{WY}*{aso_var}*tif')
            # snow_var_fns = h.fn_list(aso_dir, f'{state}/*{basin_name}*{aso_var}*tif')
            # if len(snow_var_fns) == 0:
            #     snow_var_fns = h.fn_list(aso_dir, f'{state}/*{basin_name}*/*{basin_name}*{aso_var}*tif')
            #     if len(snow_var_fns) == 0:
            #         snow_var_fns = h.fn_list(aso_dir, f'{state}/*{basin_name}*{aso_var}*tif')
            if len(snow_var_fns) == 0:
                print(f'No ASO {aso_var} files found for {basin_name} in {state} for WY {WY}. Skipping...')
                # Move to next WY in the loop
                continue
            else:
                _ = [print(f) for f in snow_var_fns]

                # Load snow var arrays and squeeze out single dimensions
                snow_var_list = [np.squeeze(xr.open_dataset(fn, chunks=chunks)) for fn in snow_var_fns]

                # Rename band_data to be more descriptive
                snow_var_list = [ds.rename_vars({'band_data': f'{band_name}'}) for ds in snow_var_list]

                # Deal with adding time input for ASO data
                inputvar = f'_{aso_var}'
                # inputvar = '.'
                snow_var_list = [np.squeeze(proc.assign_dt(ds, proc.extract_dt(fn, inputvar=inputvar))) for ds, fn in zip(snow_var_list, snow_var_fns)]

                # Get dates, could easily just pull from filenames, but this is fine
                date_list = [proc.extract_dt(fn, inputvar=inputvar)[0] for fn in snow_var_fns]

            date_list = [f.strftime('%Y%m%d') for f in date_list]
            print(date_list)

            # ASO is in EPSG 32613 for USCOBR, same as gdf_metloom. Should be good to go
            print(snow_var_list[0].rio.crs)

            # Extract ASO snow depth value with NN lookup at snotel sites
            aso_snow_point_list = []
            if not buffer:
                # Each WY has two collections
                for ds in snow_var_list:
                    # Extract data value from this point based on nearest neighbor lookup (inexact)
                    # Coords must be bracketed
                    aso_snow_point = ds[band_name].sel(x=list(gdf_metloom.geometry.x.values),
                                                    y=list(gdf_metloom.geometry.y.values),
                                                    method='nearest')
                    aso_snow_point_list.append(aso_snow_point)

                # Concatenate the list of data arrays into a single dataarray based on time dimension
                aso_snow_ts = xr.concat(aso_snow_point_list, dim='time')

                # Extract into individual time series
                aso_site_ts_list = []
                for jdx, sitenum in enumerate(sitenums):
                    # print(jdx)
                    aso_site_ts = aso_snow_ts[:, jdx, jdx]
                    aso_site_ts_list.append(aso_site_ts)

                # Generate dates for ASO data dataframe
                aso_dt_list = [snow_var['time'].values for snow_var in aso_snow_point_list]

                # Put the extracted depths into a dataframe
                aso_snowvar_ts_df = pd.DataFrame(data=np.array(aso_site_ts_list).T, columns=sitenames, index=aso_dt_list)
            else:
                # Use the date as a key for the mean values list
                aso_snow_point_dict = {}
                for ds in snow_var_list:
                    # Extract ASO snow depth values within a buffered radius of the snotel points
                    mean_vals = []
                    # Go through the sitenames and extract the mean values around a buffered point of radius r
                    for row in gdf_snotel.iterrows():
                        print(row[1].sitenames)
                        small_gdf = gpd.GeoDataFrame(data=np.array([row[1]]), columns=['sitenames', 'sitenums', 'geometry'])
                        small_site_buffer = small_gdf.buffer(distance=r)
                        try:
                            aso_clipped = ds.rio.clip(small_site_buffer)
                        except:
                            print(f'{row[1].sitenames} error \n')
                            continue
                        # print(f"Mean | Median : {aso_clipped[band_name].mean().values:.2f} | {np.nanmedian(aso_clipped[band_name]):.2f}\n")
                        mean_vals.append(float(aso_clipped[band_name].mean().values))
                        # convert strftime
                    # Add the date as a key to the dictionary with the mean values for all sites
                    aso_snow_point_dict[pd.to_datetime(ds['time'].values).strftime('%Y-%m-%d')] = mean_vals
                    # Turn into a single WY dataframe with dates as indices
                aso_snowvar_ts_df = pd.DataFrame(aso_snow_point_dict, index=sitenames).T
            aso_df_list.append(aso_snowvar_ts_df)
        aso_df = pd.concat(aso_df_list, axis=0)

In [ ]:
# Convert index to datetime
aso_df.index = pd.to_datetime(aso_df.index)
aso_df

In [ ]:
# Generate colors for plotting from palette
colors = sns.color_palette('icefire', n_colors=len(sitenames)+2)
colors = colors[:len(sitenames)-2] + colors[-2:]
colors


In [ ]:
# Plot the mean value for the 9 neighbor grid against the nn values
fig, ax = plt.subplots(figsize=(8, 5))
aso_df.plot(ax=ax, marker='d', linewidth=0, color=colors)
ax.grid('lightgray', linestyle='--', alpha=0.5)
plt.legend(bbox_to_anchor=(1, 1), frameon=False)

In [ ]:
# add both the nearest neighbor and the buffered values to the snotel gdf
site_buffer = gdf_snotel.buffer(distance=r)
gdf_snotel[f'buffered_{r}m'] = site_buffer
gdf_snotel

## Plot

In [ ]:
runbaseline = False
runhrrrmodis = True
print(runbaseline, runhrrrmodis)

In [ ]:
# Plotting params
linestyles = ['-', '--']
linewidth = 1.5

snotelcolor = 'dimgray'
isnobalcolors = ['mediumblue', 'cornflowerblue']
nwm_color = 'tomato'#'salmon'
ua_color = 'navy'#'tomato'
alpha = 0.7
isnobal_alpha = 0.6

figsize = (8, 4)
figsize = (12, 4)
figsize = (20, 4)

# Get colors from default palette
default_colors = sns.color_palette('icefire')
default_colors

In [ ]:
outdir = '/uufs/chpc.utah.edu/common/home/skiles-group3/jmhu/figures/temporal'

In [ ]:
# Update the font size parameters
plt.rcParams.update({'font.size': 18,
                     'axes.labelsize': 16,
                     'axes.titlesize': 18,
                     'xtick.labelsize': 14,
                     'ytick.labelsize': 14,
                     'legend.fontsize': 22})

# # Update the font size parameters for the boxplots
# plt.rcParams.update({'font.size': 22,
#                      'axes.labelsize': 22,
#                      'axes.titlesize': 22,
#                      'xtick.labelsize': 18,
#                      'ytick.labelsize': 18,
#                      'legend.fontsize': 24})

In [ ]:
# Update styling
# HRRR-SPIReS linestyle is -. (dashdot))
# SNOTEL linestyle is - (solid)

def identify_styles(runtype, verbose=False):
    '''Based on the input runtype, break down the relevant components and get appropriate styles'''
    if verbose:
        print(runtype)
    # colors = sns.color_palette('icefire')
    # colors = ['royalblue', 'violet', 'gold']
    # colors = ['royalblue', 'purple', 'xkcd:golden']
    colors = ['darkblue', 'mediumpurple', 'xkcd:golden']
    runtype_dict = {'baseline': ('cornflowerblue', (0, (1, 2)), 1.2, 0.8),
                    'hrrrspires': (colors[0], '-', 1.5, 1),
                    # 'snotel': ('dimgrey', '-', 2, 0.6, '|'),
                    # 'snotel': ('lightgray', '-', 0.5, 0.7, '.'),
                    'snotel': ('darkgrey', '-', 0.5, 1, '.'),
                    'nwm': (colors[1], '-', 1.5, 1),
                    # 'nwm': ('indigo', '--', 0.8, 0.8),
                    'ua': (colors[2], '-', 1.5, 1),
                    'aso_nn': ('k', 'None', 0, 1, 'o'),
                    # 'aso_grid': ('maroon', 'None', 0, 1, 'd')}
                    'aso_grid': ('orangered', 'None', 0, 1, 'd')}
    try:
        color = runtype_dict[runtype][0]
        linestyle = runtype_dict[runtype][1]
        linewidth = runtype_dict[runtype][2]
        alpha = runtype_dict[runtype][3]
        if len(runtype_dict[runtype]) > 4:
            marker = runtype_dict[runtype][4]
        else:
            marker = None
    except KeyError:
        print(f'Unknown runtype: {runtype}')
        # assign default values
        linestyle = '--'
        linewidth = 2
        alpha = 1

    return color, linestyle, linewidth, alpha, marker

In [ ]:
figsize = (16, 3)

In [ ]:
# No SDD version
fig, axes = plt.subplots(len(sitenames), 1, figsize=(figsize[0], figsize[1]*len(sitenames)), sharex=True)
for jdx, (sitenum, sitename) in enumerate(zip(sitenums, sitenames)):
    ax = axes[jdx]
    # Extract SNOTEL data for this site
    snotel_df = snotel_dfs[sitename]
    print(f'\n{sitename}')

    # Plot WY time series of snow variable
    color, linestyle, linewidth, alpha, marker = identify_styles('snotel')
    if thisvar == 'thickness':
        # (snotel_dfs[f'{sitename}']['SNOWDEPTH_m']).plot(ax=ax, label=f'{sitename} Snow Depth [m]',
        (snotel_dfs[f'{sitename}'][f'SNOWDEPTH_m']).plot(ax=ax, label='SNOTEL',
                                                        color=color, linewidth=linewidth, linestyle=linestyle, alpha=alpha,
                                                        marker=marker, markersize=3
                                                        )
        # plt.title(f'{sitename} Snow Depth')
    else:
        if snowvar == 'SWE':
            # (snotel_dfs[f'{sitename}']['SWE_m']).plot(ax=ax, label=f'{sitename} SWE [m]',
            (snotel_dfs[f'{sitename}']['SWE_m']).plot(ax=ax, label='SNOTEL',
                                                        color=color, linewidth=linewidth, linestyle=linestyle, alpha=alpha,
                                                        marker=marker, markersize=3
                                                        )
            plt.title(f'{sitename} SWE')
        else:
            # (snotel_dfs[f'{sitename}']['SNOWDENSITY_kgm3']).plot(ax=ax, label=f'{sitename} density [kgm-3]',
            (snotel_dfs[f'{sitename}']['SNOWDENSITY_kgm3']).plot(ax=ax, label='SNOTEL',
                                                            color=color, linewidth=linewidth, linestyle=linestyle, alpha=alpha,
                                                            marker=marker, markersize=3
                                                        )
            plt.title(f'{sitename} Density')

    # Plot NWM
    color, linestyle, linewidth, alpha, marker = identify_styles('nwm')
    nwm_df[sitename].plot(ax=ax, label='NWM', color=color, linewidth=linewidth, linestyle=linestyle, alpha=alpha)

    # Plot UA
    color, linestyle, linewidth, alpha, marker = identify_styles('ua')
    ua_df[sitename].plot(ax=ax, label='UA', color=color, linewidth=linewidth, linestyle=linestyle, alpha=alpha)

    # Extract model data for this site and plot on top
    if runbaseline:
        isnobal_hrrr_ts = ds_dict[f'iSnobal-HRRR_{thisvar}'][sitename]
        color, linestyle, linewidth, alpha, marker = identify_styles('baseline')
        isnobal_hrrr_ts.plot(ax=ax, label='Baseline', color=color, linewidth=linewidth, linestyle=linestyle, alpha=alpha)
    if runhrrrmodis:
        modis_hrrr_ts = ds_dict[f'HRRR-SPIReS_{thisvar}'][sitename]
        color, linestyle, linewidth, alpha, marker = identify_styles('hrrrspires')
        modis_hrrr_ts.plot(ax=ax, label='HRRR-SPIReS', color=color, linewidth=linewidth, linestyle=linestyle, alpha=alpha)

    # Plot ASO
    if noaso:
        pass
    else:
        # color, linestyle, linewidth, alpha, marker = identify_styles('aso_nn')
        color, linestyle, linewidth, alpha, marker = identify_styles('aso_grid')
        aso_df[sitenames.iloc[jdx]].plot(ax=ax,
                                    marker=marker, markersize=5,
                                    label='ASO',
                                    color=color, linewidth=linewidth, linestyle=linestyle, alpha=alpha
                                    )
    if jdx == 0:
        # Move the HRRR-SPIReS entry (-2 index) in front of the NWM and UA entries (1 and 2 indices)
        handles, labels = ax.get_legend_handles_labels()
        new_order = [0, 3, 1, 2, 4]  # Adjust based on the number of legend entries
        leg = ax.legend([handles[idx] for idx in new_order], [labels[idx] for idx in new_order],
                  bbox_to_anchor=(0.5, 1.5), frameon=True, ncol=5, loc='upper center', markerscale=4)

        # change the line width for the legend
        for line in leg.get_lines():
            line.set_linewidth(4.0)
    else:
        # turn off legend
        ax.legend().set_visible(False)

    # Manage x-ticks and labeling
    oct1 = snotel_df.index[(snotel_df.index.month == 10) & (snotel_df.index.day == 1)]
    # Do the same for April 1
    apr1 =  snotel_df.index[(snotel_df.index.month == 4) & (snotel_df.index.day == 1)]
    # Combine  DateTimeIndices into a single variable
    combined = np.concatenate([oct1, apr1])

    # set xticks for indices of choice (Oct 1 and Apr 1)
    ax.set_xticks(combined)
    # set minor xticks without labels for the 1st of every month
    ax.set_xticks(snotel_df.index[snotel_df.index.day == 1], minor=True)

    ax.annotate(sitename, xy=(0.025, 0.75), xycoords='axes fraction', ha='left',
                fontsize=20, fontstyle='italic',
                bbox=dict(boxstyle='round', facecolor='white', edgecolor='none', alpha=1)
                )
    ax.set_xlabel('')
    # Set the same ylims
    if thisvar == 'thickness':
        ax.set_ylim(0, 4)
        ax.set_ylabel('Snow Depth (m)')
    elif thisvar == 'SWE':
        ax.set_ylim(0, 1.5)
        ax.set_ylabel('SWE [m]')
    else:
        ax.set_ylim(0, 1000)
    ax.grid('lightgray', linestyle='--', alpha=0.5)
# Save figure to file
# out_fn = f'{outdir}/{basin}_snotel_por_{var}_comparison.png'
out_fn = f'{outdir}/{basin}_snotel_por_{var}_comparison_manuscript_size.png'

# if os.path.exists(out_fn):
#     print(f'{out_fn} already exists, skipping save.')
# else:
print(f'Saving figure to {out_fn}')
plt.savefig(out_fn, bbox_inches='tight', dpi=300)

In [ ]:
# calculate percent of peak for each water year for each product, using SNOTEL as a reference
def calc_percent_peak(df, snotel_df, peak_col, label):
    '''Calculate peak depth and percent of snotel for each water year
    Uses groupby index (calendar year) but this should be fine as we expect peak
    to be in the springtime
    '''
    # Get the peak value for each water year
    peak_values = df.groupby(df.index.year)[peak_col].max()
    # Get the dates of the peak values
    peak_dates = df.groupby(df.index.year)[peak_col].idxmax()

    # Calculate the reference SNOTEL peak value and date for each WY
    snotel_peak_values = snotel_df.groupby(snotel_df.index.year)['SNOWDEPTH_m'].max()
    snotel_peak_dates = snotel_df.groupby(snotel_df.index.year)['SNOWDEPTH_m'].idxmax()

    # modify everything by skipping the first calendar year (just the fall)
    peak_values = peak_values.iloc[1:]
    peak_dates = peak_dates.iloc[1:]
    snotel_peak_values = snotel_peak_values.iloc[1:]
    snotel_peak_dates = snotel_peak_dates.iloc[1:]

    # Calculate percent of peak for each value
    percent_peak = peak_values / snotel_peak_values * 100

    # Print the peak values, dates, and percent of peak
    for year in peak_values.index:
        print(f'WY{year}, Peak: {peak_values[year]:.2f} [{snotel_peak_values[year]:.2f}], '
              f'Peak Date: {peak_dates[year].strftime('%Y-%m-%d')} [{snotel_peak_dates[year].strftime('%Y-%m-%d')}], '
              f'{percent_peak[year]:.1f}%')

    # Put it into a df
    snowpercent_peak = pd.DataFrame({
        f'{label}_Peak': peak_values,
        f'{label}_Peak_Date': peak_dates,
        f'{label}_SNOTEL_Peak': snotel_peak_values,
        f'{label}_SNOTEL_Peak_Date': snotel_peak_dates,
        f'{label}_percent': percent_peak
    })
    return snowpercent_peak

# Only do this if the var is depth or SWE
if thisvar in ['thickness', 'SWE']:
    peak_dict = dict()
    for jdx, (sitenum, sitename) in enumerate(zip(sitenums, sitenames)):
        print(f'\n{sitename}')
        # Extract SNOTEL data for this site
        snotel_df = snotel_dfs[sitename]

        print('\nHRRR-SPIReS')
        hs_df = calc_percent_peak(df=ds_dict[f'HRRR-SPIReS_{thisvar}'], snotel_df=snotel_df, peak_col=sitename, label='HRRR-SPIReS')
        print('\nNWM')
        nwm_peak_df = calc_percent_peak(df=nwm_df, snotel_df=snotel_df, peak_col=sitename, label='NWM')
        print('\nUA')
        ua_peak_df = calc_percent_peak(df=ua_df, snotel_df=snotel_df, peak_col=sitename, label='UA')

        peak_df = pd.concat([hs_df, nwm_peak_df, ua_peak_df], axis=1)
        peak_dict[sitename] = peak_df

In [ ]:
# Only do this if the var is depth or SWE
if thisvar in ['thickness', 'SWE']:
    # Turn the dict into a giant dataframe with multilevel indexing
    # Use the sitename (keys) as the top level in the index
    big_peak_df = pd.concat(peak_dict.values(), keys=peak_dict.keys(), axis=0)
    big_peak_df

In [ ]:
# Only do this if the var is depth or SWE
if thisvar in ['thickness', 'SWE']:
    # Write this to file if it does not exist
    out_fn = f'{outdir}/{basin}_snotel_por_{var}_peak_comparison.csv'
    if not os.path.exists(out_fn):
        print(f'Saving to {out_fn}')
        big_peak_df.to_csv(out_fn)

In [ ]:
# Only do this if the var is depth or SWE
if thisvar in ['thickness', 'SWE']:
    # Get the stats
    # basin = 'blue'
    stats_fn = f'{outdir}/{basin}_snotel_por_{var}_peak_comparison.csv'
    big_peak_df = pd.read_csv(stats_fn, index_col=0)
    stats_df = pd.read_csv(stats_fn, index_col=0)

    # Filter out the percents
    # Get the median value by colummn
    print(f'{stats_df.filter(like='percent').median()}\n')

    # Get the IQR of the percents
    print('IQR top:')
    print(f'{stats_df.filter(like='percent').quantile(0.75)}')
    print('\nIQR bottom:')
    print(f'{stats_df.filter(like='percent').quantile(0.25)}')

    # and the diffs
    print('IQR span:')
    print(f'{stats_df.filter(like='percent').quantile(0.75) - stats_df.filter(like='percent').quantile(0.25)}')

    # # And now compare the timing? Compute the difference in days between the peak dates? No, too noisy for this
    # stats_df.filter(like='Date')
    # stats_df['HRRR-SPIReS_Peak_Date'] = pd.to_datetime(stats_df['HRRR-SPIReS_Peak_Date']).dt.date
    # stats_df['NWM_Peak_Date'] = pd.to_datetime(stats_df['NWM_Peak_Date']).dt.date
    # stats_df['UA_Peak_Date'] = pd.to_datetime(stats_df['UA_Peak_Date']).dt.date
    # stats_df['HRRR-SPIReS_SNOTEL_Peak_Date'] = pd.to_datetime(stats_df['HRRR-SPIReS_SNOTEL_Peak_Date']).dt.date
    # stats_df['NWM_SNOTEL_Peak_Date'] = pd.to_datetime(stats_df['NWM_SNOTEL_Peak_Date']).dt.date
    # stats_df['UA_SNOTEL_Peak_Date'] = pd.to_datetime(stats_df['UA_SNOTEL_Peak_Date']).dt.date
    # stats_df['HRRR-SPIReS_Peak_Date_diff'] = stats_df['HRRR-SPIReS_Peak_Date'] - stats_df['HRRR-SPIReS_SNOTEL_Peak_Date']
    # stats_df['NWM_Peak_Date_diff'] = stats_df['NWM_Peak_Date'] - stats_df['NWM_SNOTEL_Peak_Date']
    # stats_df['UA_Peak_Date_diff'] = stats_df['UA_Peak_Date'] - stats_df['UA_SNOTEL_Peak_Date']
    # # stats_df
    # stats_df.filter(like='Date')

In [ ]:
# Only do this if the var is depth or SWE
if thisvar in ['thickness', 'SWE']:
    # Drop all of the columns except for the percent ones
    percent_df = big_peak_df.filter(like='percent')
    percent_df

In [ ]:
# Only do this if the var is depth or SWE
if thisvar in ['thickness', 'SWE']:
    # Update the font size parameters for the boxplots
    plt.rcParams.update({'font.size': 22,
                        'axes.labelsize': 22,
                        'axes.titlesize': 22,
                        'xtick.labelsize': 18,
                        'ytick.labelsize': 18,
                        'legend.fontsize': 24})

    # Plot this as boxplots for the basin
    fig, ax = plt.subplots(figsize=(6, 4))
    agg = False
    # Mark out the 100% line
    ax.axhline(y=100, color='darkgrey', linestyle='--', linewidth=2, alpha=0.5, label='SNOTEL Peak')
    # Group by the first level of the index (sitenames) and plot the mean percent values as a boxplot and use different face colors for each boxplot
    if agg:
        # this one plots the mean of each site across water years
        ax, props = percent_df.groupby(level=0).mean().round(0).plot(kind='box', ax=ax, patch_artist=True, return_type='both')
    else:
        # this one plots each year and site for the full distribution
        ax, props = percent_df.plot(kind='box', ax=ax, patch_artist=True, return_type='both')

    # colors_cm = sns.color_palette('icefire', n_colors=3)
    # colors_cm = ['royalblue', 'purple', 'xkcd:golden']
    colors_cm = ['darkblue', 'mediumpurple', 'xkcd:golden']
    for patch, color in zip(props['boxes'], colors_cm):
        patch.set_facecolor(color)
        patch.set_edgecolor('black')
    # Change the median line color
    for median in props['medians']:
        median.set_color('whitesmoke')
    for whisk, cap in zip(props['whiskers'], props['caps']):
        whisk.set_color('black')
        cap.set_color('black')

    ax.grid('lightgray', linestyle='--', alpha=0.5)
    # Remove _percent from each x-tick label
    ax.set_xticklabels([label.get_text().replace('_percent', '') for label in ax.get_xticklabels()])
    # Add % to each y-tick label
    ax.set_yticklabels([f'{int(label.get_text())}%' for label in ax.get_yticklabels()])
    ax.set_title(f'{basin.capitalize()} Peak {var.capitalize()}')
    ax.set_ylabel('% SNOTEL Peak')
    if agg:
        ax.set_ylim(50, 130)
    else:
        ax.set_ylim(40, 150)
    # # Mark out the 100% line
    # ax.axhline(y=100, color='darkgrey', linestyle='--', linewidth=2, alpha=0.5, label='SNOTEL Peak')
    # Save to file
    if agg:
        out_fn = f'{outdir}/{basin}_snotel_por_{var}_percent_peak.png'
    else:
        out_fn = f'{outdir}/{basin}_snotel_por_{var}_percent_peak_disaggregated.png'
    print(f'Saving figure to {out_fn}')
    plt.savefig(out_fn, bbox_inches='tight', dpi=300)
    print(percent_df.shape[0])

In [ ]:
# Update styling for difference plots
def identify_styles_diff(runtype, verbose=False):
    '''Based on the input runtype, break down the relevant components and get appropriate styles'''
    if verbose:
        print(runtype)
    default_colors = sns.color_palette('icefire')
    runtype_dict = {'baseline': ('cornflowerblue', (0, (1, 2)), 1.2, 0.8),
                    'hrrrspires': (default_colors[1], '-', 1.5, 1),
                    'snotel': ('lightgray', '-', 0.5, 0.7, '.'),
                    'ua': (default_colors[-1], 'None', 1.5, 1, '.'),
                    'nwm': (default_colors[-3], 'None', 1, 1, 'x'),
                    'aso_nn': ('maroon', 'None', 0, 1, 'o'),
                    'aso_grid': ('maroon', 'None', 0, 1, 'x')}
    try:
        color = runtype_dict[runtype][0]
        linestyle = runtype_dict[runtype][1]
        linewidth = runtype_dict[runtype][2]
        alpha = runtype_dict[runtype][3]
        if len(runtype_dict[runtype]) > 4:
            marker = runtype_dict[runtype][4]
        else:
            marker = None
    except KeyError:
        print(f'Unknown runtype: {runtype}')
        # assign default values
        linestyle = '--'
        linewidth = 2
        alpha = 1

    return color, linestyle, linewidth, alpha, marker

In [ ]:
if snowvar == 'SNOWDEPTH':
    snowvar = 'SNOWDEPTH_m'  # SNOTEL snow depth variable
    # Disappearance date parameters
    snow_name = 'Snow Depth'
elif snowvar == 'SWE':
    snowvar = 'SWE_m' # SNOTEL SWE variable
elif snowvar == 'both':
    snowvar = 'SNOWDENSITY_kgm3' # SNOTEL snow density variable

In [ ]:
# Plot the differences from SNOTEL measurement
figsize = (16, 2.5)
fig, axes = plt.subplots(len(sitenames), 1, figsize=(figsize[0], figsize[1]*len(sitenames)), sharex=True)
for jdx, (sitenum, sitename) in enumerate(zip(sitenums, sitenames)):
    ax = axes[jdx]
    # Extract SNOTEL data for this site
    snotel_df = snotel_dfs[sitename][f'{snowvar}'].resample('D').mean()
    print(f'\n{sitename}')

    # Extract model data for this site and resample to daily mean to match SNOTEL precisely
    modis_hrrr_ts = ds_dict[f'HRRR-SPIReS_{thisvar}'][sitename].resample('D').mean()
    nwm_ts = nwm_df[sitename].resample('D').mean()
    ua_ts = ua_df[sitename].resample('D').mean()

    # Make indices tz aware
    modis_hrrr_ts.index = modis_hrrr_ts.index.tz_localize('UTC')
    nwm_ts.index = nwm_ts.index.tz_localize('UTC')
    ua_ts.index = ua_ts.index.tz_localize('UTC')

    # Compute and plot the differences
    color, linestyle, linewidth, alpha, marker = identify_styles_diff('hrrrspires')
    (modis_hrrr_ts - snotel_df).plot(ax=ax, label='HRRR-SPIReS diff', color=color, linewidth=linewidth, linestyle=linestyle, alpha=alpha)

    # Plot NWM
    color, linestyle, linewidth, alpha, marker = identify_styles_diff('nwm')
    (nwm_ts - snotel_df).plot(ax=ax, label='NWM diff', color=color, linewidth=linewidth, linestyle=linestyle, alpha=alpha, marker=marker, markersize=1)
    # Plot UA
    color, linestyle, linewidth, alpha, marker = identify_styles_diff('ua')
    (ua_ts - snotel_df).plot(ax=ax, label='UA diff', color=color, linewidth=linewidth, linestyle=linestyle, alpha=alpha, marker=marker, markersize=1)

    # # Shade above and below zero different colors
    # ax.fill_between((modis_hrrr_ts - snotel_df).index, (modis_hrrr_ts - snotel_df), 0, where=(modis_hrrr_ts - snotel_df) > 0, alpha=0.5)
    # ax.fill_between((modis_hrrr_ts - snotel_df).index, (modis_hrrr_ts - snotel_df), 0, where=(modis_hrrr_ts - snotel_df) < 0, color='salmon', alpha=0.5)

    # Set title as SNOTEL site
    ax.set_title(f'{sitename}')

    # Set the same ylims
    if thisvar == 'thickness':
        ax.set_ylim(-1.5, 1.5)
    elif thisvar == 'SWE':
        ax.set_ylim(-0.5, 0.5)
    else:
        ax.set_ylim(-150, 150)
    # add gridlines
    ax.grid('lightgray', linestyle='--', alpha=0.5)

    # Add zero line
    ax.axhline(0, color='k', linestyle='-', linewidth=1.5)

    # Set the x and y labels
    ax.set_xlabel('')
    units = snowvar.split('_')[1]
    ax.set_ylabel(f'Diff from SNOTEL [{units}]')

    # Shade november up to april of each year
    for year in WYs:
        if year == WYs[-1]:
            ax.axvspan(pd.Timestamp(year=year-1, month=11, day=1), pd.Timestamp(year=year, month=4, day=1), color='lightgray', alpha=0.3, label='Accumulation season [Nov-Apr]')
        else:
            ax.axvspan(pd.Timestamp(year=year-1, month=11, day=1), pd.Timestamp(year=year, month=4, day=1), color='lightgray', alpha=0.3)

    # Handle legend
    if jdx == 0:
        ax.legend(bbox_to_anchor=(1,1), alignment='center')
    else:
        # turn off legend
        ax.legend().set_visible(False)

# Save figure to file
out_fn = f'{outdir}/{basin}_snotel_por_diff_{var}_comparison.png'
# if os.path.exists(out_fn):
#     print(f'{out_fn} already exists, skipping save.')
# else:
print(f'Saving figure to {out_fn}')
plt.savefig(out_fn, bbox_inches='tight', dpi=300)

In [ ]:
# Plot the differences from SNOTEL measurement as a percentage of the SNOTEL measurement
# Plot the differences from SNOTEL measurement
figsize = (16, 2.5)
fig, axes = plt.subplots(len(sitenames), 1, figsize=(figsize[0], figsize[1]*len(sitenames)), sharex=True)
for jdx, (sitenum, sitename) in enumerate(zip(sitenums, sitenames)):
    ax = axes[jdx]
    # Extract SNOTEL data for this site
    snotel_df = snotel_dfs[sitename][f'{snowvar}'].resample('D').mean()
    print(f'\n{sitename}')

    # Extract model data for this site and resample to daily mean to match SNOTEL precisely
    modis_hrrr_ts = ds_dict[f'HRRR-SPIReS_{thisvar}'][sitename].resample('D').mean()
    nwm_ts = nwm_df[sitename].resample('D').mean()
    ua_ts = ua_df[sitename].resample('D').mean()

    # Make indices tz aware
    modis_hrrr_ts.index = modis_hrrr_ts.index.tz_localize('UTC')
    nwm_ts.index = nwm_ts.index.tz_localize('UTC')
    ua_ts.index = ua_ts.index.tz_localize('UTC')

    # Compute and plot the differences
    color, linestyle, linewidth, alpha, marker = identify_styles_diff('hrrrspires')
    ((modis_hrrr_ts - snotel_df)/snotel_df * 100).plot(ax=ax, label='HRRR-SPIReS diff', color=color, linewidth=linewidth, linestyle=linestyle, alpha=alpha)

    # Plot NWM
    color, linestyle, linewidth, alpha, marker = identify_styles_diff('nwm')
    ((nwm_ts - snotel_df)/snotel_df * 100).plot(ax=ax, label='NWM diff', color=color, linewidth=linewidth, linestyle=linestyle, alpha=alpha, marker=marker, markersize=1)
    # Plot UA
    color, linestyle, linewidth, alpha, marker = identify_styles_diff('ua')
    ((ua_ts - snotel_df)/snotel_df * 100).plot(ax=ax, label='UA diff', color=color, linewidth=linewidth, linestyle=linestyle, alpha=alpha, marker=marker, markersize=1)

    # # Shade above and below zero different colors
    # ax.fill_between((modis_hrrr_ts - snotel_df).index, (modis_hrrr_ts - snotel_df), 0, where=(modis_hrrr_ts - snotel_df) > 0, alpha=0.5)
    # ax.fill_between((modis_hrrr_ts - snotel_df).index, (modis_hrrr_ts - snotel_df), 0, where=(modis_hrrr_ts - snotel_df) < 0, color='salmon', alpha=0.5)

    # Set title as SNOTEL site
    ax.set_title(f'{sitename}')

    # Set the same ylims
    if thisvar == 'thickness':
        ax.set_ylim(-150, 150)
    elif thisvar == 'SWE':
        ax.set_ylim(-150, 150)
    else:
        ax.set_ylim(-100, 100)
    # add gridlines
    ax.grid('lightgray', linestyle='--', alpha=0.5)

    # Add zero line
    ax.axhline(0, color='k', linestyle='-', linewidth=1.5)

    # Set the x and y labels
    ax.set_xlabel('')
    ax.set_ylabel('Diff from SNOTEL [%]')

    # Shade november through april of each year
    for year in WYs:
        if year == WYs[-1]:
            ax.axvspan(pd.Timestamp(year=year-1, month=11, day=1), pd.Timestamp(year=year, month=4, day=30), color='lightgray', alpha=0.3, label='Accumulation season [Nov-Apr]')
        else:
            ax.axvspan(pd.Timestamp(year=year-1, month=11, day=1), pd.Timestamp(year=year, month=4, day=30), color='lightgray', alpha=0.3)

    # Handle legend
    if jdx == 0:
        ax.legend(bbox_to_anchor=(1,1), alignment='center')
    else:
        # turn off legend
        ax.legend().set_visible(False)

# Save figure to file
out_fn = f'{outdir}/{basin}_snotel_por_percentdiff_{var}_comparison.png'
# if os.path.exists(out_fn):
#     print(f'{out_fn} already exists, skipping save.')
# else:
print(f'Saving figure to {out_fn}')
plt.savefig(out_fn, bbox_inches='tight', dpi=300)

#### Calculate snow disappearance date separately to plot and re-plot quickly

In [ ]:
big_sdd_list = []
for WY in WYs:
    verbose = False

    sdd_list = []
    for jdx, (sitenum, sitename) in enumerate(zip(sitenums, sitenames)):
        # Extract SNOTEL data for this site
        snotel_df = snotel_dfs[sitename]
        print(f'\n{sitename}')

        # Extract model data for this site
        if runbaseline:
            isnobal_hrrr_ts = ds_dict[f'iSnobal-HRRR_{thisvar}'][sitename]
        if runhrrrmodis:
            modis_hrrr_ts = ds_dict[f'HRRR-SPIReS_{thisvar}'][sitename]

        # Calculate disappearance dates
        snotel_sdd, _ = proc.calc_sdd(snotel_df[f'{snowvar}'],
                                    verbose=verbose)
        print(f'SNOTEL: {snotel_sdd}')

        # Convert model data to Pandas Series
        if runbaseline:
            classic_sdd, _ = proc.calc_sdd(pd.Series(data=np.squeeze(isnobal_hrrr_ts), index=isnobal_hrrr_ts.index),
                                        verbose=verbose)
            print(f'Baseline: {classic_sdd}')

        # Convert model data to Pandas Series
        if runhrrrmodis:
            modis_hrrr_sdd, _ = proc.calc_sdd(pd.Series(data=np.squeeze(modis_hrrr_ts), index=modis_hrrr_ts.index),
                                        verbose=verbose)
            print(f'HRRR-MODIS: {modis_hrrr_sdd}')

        try:
            nwm_sdd, _ = proc.calc_sdd(pd.Series(data=np.squeeze(nwm_daily_df[sitename].values), index=nwm_daily_df.index),
                                            verbose=verbose)
            print(f'NWM: {nwm_sdd}')
        except:
            print("Something wrong with NWM extract")
            # add default day
            nwm_sdd = pd.Timestamp(year=WY, month=12, day=25)

        try:
            ua_sdd, _ = proc.calc_sdd(pd.Series(data=np.squeeze(ua_datadf[sitename].values), index=ua_datadf.index),
                                            verbose=verbose)
            print(f'UA: {ua_sdd}')
        except:
            print("Something wrong with UA extract")
            # add default day
            nwm_sdd = pd.Timestamp(year=WY, month=12, day=25)


        if runbaseline and runhrrrmodis:
            sdd_list.append([snotel_sdd, classic_sdd, modis_hrrr_sdd, nwm_sdd, ua_sdd])
        else:
            if runbaseline:
                sdd_list.append([snotel_sdd, classic_sdd, nwm_sdd, ua_sdd])
            if runhrrrmodis:
                sdd_list.append([snotel_sdd, modis_hrrr_sdd, nwm_sdd, ua_sdd])
    big_sdd_list.append(sdd_list)

In [ ]:
def calc_lrm_stats(model, obs):
    # Compute a line of best fit
    m, b = np.polyfit(obs, model, 1)

    # Add the fit equation to the plot
    fit_eq = f'y = {m:.2f}x + {b:.2f}'
    # Add the correlation coefficient
    r = np.corrcoef(obs, model)[0, 1]
    fit_eq += f'\nr = {r:.2f}'

    # Add the coefficient of determination R2
    r_squared = r ** 2

    # Add the number of points
    n_points = len(obs)

    return r, r_squared, n_points

def calculate_kge(r, model, obs):
    '''Calculate Kling-Gupta Efficiency (KGE) between two depth arrays'''
    # KGE is calculated as:
    # KGE = 1 - sqrt((r - 1)^2 + (alpha - 1)^2 + (beta - 1)^2)
    # where r is the correlation coefficient, alpha is the ratio of the standard deviations,
    # and beta is the ratio of the means.
    kge = 1 - np.sqrt((r - 1) ** 2 + (np.std(model) / np.std(obs) - 1) ** 2 + (np.mean(model) / np.mean(obs) - 1) ** 2)
    return kge

# Calculate RMSE first
def calculate_rmse(model, obs):
    '''Calculate RMSE between two depth arrays'''
    rmse = np.sqrt(np.nanmean((model - obs) ** 2))
    return rmse

# Now based on a defined normalization factor, calculate the normalized RMSE
def calculate_nrmse(model, obs, normalization_factor):
    '''Calculate normalized RMSE between two depth arrays
    Normalization factors: range of ASO depth, mean ASO depth, standard deviation of ASO depth
    '''
    rmse = calculate_rmse(model, obs)
    if normalization_factor == 'range':
        normalization_factor = np.nanmax(obs) - np.nanmin(obs)
    elif normalization_factor == 'mean':
        normalization_factor = np.nanmean(obs)
    elif normalization_factor == 'std':
        normalization_factor = np.nanstd(obs)
    nrmse = rmse / normalization_factor
    return nrmse

def calculate_mae(model, obs):
    '''Calculate Mean Absolute Error between two depth arrays'''
    mae = np.nanmean(np.abs(model - obs))
    return mae

In [ ]:
# Extract sitenames, replacing spaces, parentheses and other symbols with underscores
fixed_names  = [sitename.replace(' ', '_').replace('(', '').replace(')', '').replace('-', '_') for sitename in sitenames]
fixed_names

In [ ]:
def remove_empty_dates(model, obs):
    '''Removes corresponding entries missing in either model or obs'''
    # Identify dates where model is nan
    model_nan_dates = model[model.isna()].index
    # Identify dates where obs is nan
    obs_nan_dates = obs[obs.isna()].index
    # Combine the two sets of dates
    nan_dates = model_nan_dates.union(obs_nan_dates)
    # Remove the nan dates from both model and obs
    model_cleaned = model.drop(nan_dates)
    obs_cleaned = obs.drop(nan_dates)
    # Remove any dates where the model value is infinite
    model_infinite_dates = model_cleaned[model_cleaned.isin([np.inf, -np.inf])].index
    obs_infinite_dates = obs_cleaned[obs_cleaned.isin([np.inf, -np.inf])].index
    # Combine the two sets of infinite dates
    infinite_dates = model_infinite_dates.union(obs_infinite_dates)
    # Remove the infinite dates from both model and obs
    model_cleaned = model_cleaned.drop(infinite_dates)
    obs_cleaned = obs_cleaned.drop(infinite_dates)
    return model_cleaned, obs_cleaned

def trim_series(model, obs):
    '''Trim the model and observation series to the same length'''
    start_date = max(model.index.min(), obs.index.min())
    end_date = min(model.index.max(), obs.index.max())
    model_trimmed = model.loc[start_date:end_date]
    obs_trimmed = obs.loc[start_date:end_date]
    model_cleaned, obs_cleaned = remove_empty_dates(model_trimmed, obs_trimmed)
    return model_cleaned, obs_cleaned

In [ ]:
# Compute RMSE for each model against SNOTEL
# Plot the differences from SNOTEL measurement as a percentage of the SNOTEL measurement
# Plot the differences from SNOTEL measurement
stats_df_list = []
for jdx, (sitenum, sitename) in enumerate(zip(sitenums, sitenames)):
    print(f'\n{sitename}')
    fixed_name = fixed_names[jdx]
    # Extract SNOTEL data for this site
    snotel_df = snotel_dfs[sitename][f'{snowvar}'].resample('D').mean()

    # Extract model data for this site and resample to daily mean to match SNOTEL precisely
    isnobal_ts = ds_dict[f'HRRR-SPIReS_{thisvar}'][sitename].resample('D').mean()
    nwm_ts = nwm_df[sitename].resample('D').mean()
    ua_ts = ua_df[sitename].resample('D').mean()

    # Make indices tz aware
    isnobal_ts.index = isnobal_ts.index.tz_localize('UTC')
    nwm_ts.index = nwm_ts.index.tz_localize('UTC')
    ua_ts.index = ua_ts.index.tz_localize('UTC')

    # Compute temporal SNOTEL statistics
    # And store them into a dictionary
    stats_dict = {}
    for model_ts, runtype in zip([isnobal_ts, nwm_ts, ua_ts], ['HRRR-SPIReS', 'NWM', 'UA']):
        # Trim each time series to the minimum of the snotel or model range
        model, obs = trim_series(model_ts, snotel_df)
        r, r_squared, n_points = calc_lrm_stats(model=model, obs=obs)
        rmse = calculate_rmse(model=model, obs=obs)
        kge = calculate_kge(model=model, obs=obs, r=r)
        nrmse = calculate_nrmse(model=model, obs=obs, normalization_factor='range')
        mae = calculate_mae(model=model, obs=obs)
        # print(f'{runtype} RMSE: {rmse:.2f} m, KGE: {kge:.2f}')
        stats_dict[f'{fixed_name}_{runtype}'] = {
            'n_points': n_points,
            'mae': mae,
            'rmse': rmse,
            'nrmse': nrmse,
            'r': r,
            'r_squared': r_squared,
            'kge': kge
        }
    # Convert to dataframe
    stats_df = pd.DataFrame(stats_dict)

    # Append to list
    stats_df_list.append(stats_df)

# Generate a single dataframe with all stats
stats_df = pd.concat(stats_df_list, axis=1)

# Rearrange stats_df into a multiindex dataframe
# Group by sitenames and runtype
stats_df = stats_df.T
stats_df = stats_df.reset_index()
# Split the index column into sitenames and runtype using the last underscore as the separator
stats_df[['sitenames', 'runtype']] = stats_df['index'].str.rsplit('_', n=1, expand=True)
# Drop the 'index' column
stats_df.drop(columns=['index'], inplace=True)
# Set multiindex
stats_df.set_index(['sitenames', 'runtype'], inplace=True)
# Sort by sitenames and runtype
stats_df.sort_index(inplace=True)

# And save it to file
out_stats_fn = f'{outdir}/{basin}_snotel_por_{var}_model_eval_stats.csv'
# if os.path.exists(out_stats_fn):
#     print(f'{out_stats_fn} already exists, skipping save.')
# else:
print(f'Saving stats to {out_stats_fn}')
stats_df.to_csv(out_stats_fn)
stats_df


In [ ]:
annotate = False
if basin == 'yampa':
    figsize = (18, 14)
    ymax = 0.42
elif basin == 'animas':
    figsize = (9 * len(sitenames)/4, 13)
    ymax = 0.3
else:
    figsize = (9, 13)
    ymax = 0.3
fig, axa = plt.subplots(3, 1, figsize=figsize, sharex=True)
for jdx, stat in enumerate(['mae', 'nrmse', 'kge']):
    ax = axa[jdx]
    ylab = [f'Mean Absolute Error {units}', 'Normalized RMSE', 'Kling-Gupta Efficiency'][jdx]
    yshift = [0.015, 0.015, 0.04][jdx]
    if var == 'thickness' or var == 'depth':
        ylims = [(0, ymax), (0, ymax), (0, 1)][jdx]
    elif var == 'SWE':
        ylims = [(0, ymax), (0, ymax), (0, 1)][jdx]
    else:
        ylims = [(0, 150), (0, ymax), (0, 1)][jdx]

    # Plot MAE as barplot, coloring bars by runtype (HRRR-SPIReS, NWM, UA)
    stats_df[stat].unstack().plot(kind='bar', ax=ax,
                                  width=0.63,
                                #   color=[default_colors[1], default_colors[3], default_colors[-1]],
                                  color=['darkblue', 'mediumpurple', 'xkcd:golden'],
                                  legend=True, rot=0)

    if annotate:
        # Get the x locations of each bar group center
        x_locations = np.arange(len(stats_df.index.unique('sitenames')))

        # annotate each bar within each bar group
        for i, sitename in enumerate(stats_df.index.unique('sitenames')):
            for j, runtype in enumerate(stats_df.loc[sitename].index):
                value = stats_df.loc[sitename, runtype][stat]
                ax.text(x_locations[i] + (j-1) * 0.21, value - yshift, f'{value:.2f}', ha='center', va='bottom', color='w', fontsize=9)

    ax.set_xlabel('')
    # Rotate xtick labels
    ax.set_xticklabels([label.get_text().replace('_', '\n') for label in ax.get_xticklabels()], fontsize=16, ha='center')
    ax.set_ylabel(ylab)
    ax.set_ylim(ylims)
    # Set gridlines
    ax.grid('lightgray', linestyle='--', alpha=0.5)
    # Move legend outside the plot and turn off for all except the first plot
    if jdx != 0:
        ax.legend().set_visible(False)
    else:
        ax.legend(title='Runtype', bbox_to_anchor=(1, 1), loc='upper left')

plt.suptitle(f'{basin.capitalize()} SNOTEL period of record {var} evaluation')
plt.tight_layout()

# Save to file
out_fn = f'{outdir}/{basin}_snotel_por_{var}_model_eval_stats.png'
print(f'Saving figure to {out_fn}')
plt.savefig(out_fn, bbox_inches='tight', dpi=300)

# Read in all the stats and plot up a combined KGE figure

In [ ]:
# Load all the csv files with stats
csv_fns = [h.fn_list(outdir, f'{basin}_snotel_por_{var}_model_eval_stats.csv')[0] for basin in ['blue', 'animas', 'yampa']]
df_list = [pd.read_csv(csv) for csv in csv_fns]
# Extract the basin name from the filename and store in another column
for df, csv_fn in zip(df_list, csv_fns):
    basin_name = os.path.basename(csv_fn).split('_')[0]  # Assuming the basin name is the first part of the filename
    df['basin'] = basin_name

# Concatenate all dataframes into a single dataframe
big_stat_df = pd.concat(df_list, axis=0)

# Generate a multiindex dataframe with sitenames, runtype, and basin
big_stat_df = big_stat_df.set_index(['basin', 'sitenames', 'runtype'])
big_stat_df

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
big_stat_df['kge'].plot(kind='bar', width=0.8, ax=ax)
# big_stat_df['kge']['blue']['Hoosier_Pass_531']

In [ ]:
big_stat_df['kge'].unstack().loc['blue'].loc['Hoosier_Pass_531']

In [ ]:
big_stat_df['kge'].unstack().mean().mean()

In [ ]:
# Plot KGE for each basin and sitename
fig, ax = plt.subplots(figsize=(12, 4))
big_stat_df['kge'].unstack().plot(kind='bar', ax=ax, width=0.8,
                                #    color=[default_colors[1], default_colors[3], default_colors[-1]],
                                   color=['darkblue', 'mediumpurple', 'xkcd:golden'],
                                   legend=True, rot=0)
# # remove basin name from xtick labels
# ax.set_xticklabels([sitename.replace('_', ' ') for sitename in big_stat_df.index.unique('sitenames')], rotation=45, ha='right')
ax.set_xticklabels([sitename.replace('_', ' ') for sitename in big_stat_df['kge'].unstack().index.get_level_values('sitenames')], rotation=45, ha='right', fontsize=10)
# ax.set_xticks([])
ax.set_xlabel('')
ax.set_ylabel('Kling-Gupta Efficiency', fontsize=16)
# Increase ytick label fontsize
ax.tick_params(axis='y', labelsize=14)
# Add hline at 0.41 special linestyle with spaced dots
ax.axhline(0.41, color='k', linestyle=(0, (1, 5)), linewidth=1, alpha=1)
# Add another at the overall average KGE value
ax.axhline(big_stat_df['kge'].unstack().mean().mean(), color='k', linestyle=(0, (1, 5)), linewidth=1, alpha=1)
plt.legend(bbox_to_anchor=(0.5, 1.1), ncol=3, loc='center', title='', fontsize=12)
plt.savefig(f'{outdir}/all_basins_snotel_por_{var}_kge_comparison.svg', bbox_inches='tight', dpi=300)